<h1 style="color:#1B4F72; font-weight:bold;text-align:center;">
Stage 1 Modeling for Single-Gas vs Mixture Classification
</h1>

<h2 style="color:#1B4F72; font-weight:bold;">
1. Introduction
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This notebook presents the Stage 1 modeling phase of the time-series gas sensor analysis pipeline. At this stage, machine learning 
models are developed and evaluated using the engineered features extracted in the previous steps. The primary objective is 
to classify sensor signal windows into <b>single-gas</b> and <b>mixture-gas</b> exposure categories while ensuring a robust 
and leakage-aware evaluation strategy.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Gas sensor signals are typically noisy, high-dimensional, and strongly time-dependent, which makes raw measurements less 
suitable for direct modeling. For this reason, the preprocessing and feature engineering stages were used to transform the 
original time-series signals into a compact set of informative descriptors, including statistical, temporal, shape-based, 
and variability features. <i>(Literature Review, Section 3.2, Feature Engineering for Gas Sensor Time-Series)</i>
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
These engineered features are designed to capture complementary aspects of the sensor response, including magnitude, temporal dynamics, 
and structural patterns. Such characteristics are expected to differ between single-gas and mixture exposures, making them suitable for classification tasks.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A key concern in this stage is to obtain reliable and generalizable performance estimates. Since multiple windows may come 
from the same experimental run, the evaluation must be designed carefully to avoid data leakage. All modeling steps are performed 
under a strict <b>run-aware evaluation design</b>, ensuring that windows originating from the same run are kept within the same subset 
during training and testing. <i>(Literature Review, Section 5, Evaluation Strategy for Gas Sensor Classification)</i>
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The notebook follows a structured workflow: dataset preparation, target definition, leakage-aware train-test splitting, 
baseline model training, and performance comparison. An optional dimensionality reduction step (e.g., PCA) may be explored 
to assess the impact of feature redundancy on model performance.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.1 Modeling Objective
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The objective of this notebook is to build and compare machine learning models for <b>Stage 1</b> of the hierarchical gas classification pipeline. 
At this stage, each feature-engineered window is assigned to one of two classes: <code>single_gas</code> or <code>mixture</code>.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The binary target label is derived from the experiment metadata, where each window inherits its class based on the original experimental 
condition of the run.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This decision is operationally important because it determines whether a sample should later enter the Stage 2 single-gas classification branch. 
The goal is therefore not only to achieve strong classification performance, but also to obtain a model whose behavior is reliable enough to support downstream routing.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.2 CRISP-DM Context
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Within CRISP-DM, this notebook belongs mainly to the <b>Modeling</b> and <b>Evaluation</b> phases. It uses the feature-engineered representation 
created in the previous notebook and evaluates how effectively different models can separate single-gas and mixture responses under a leakage-safe design.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The expected outcome of this phase is a justified Stage 1 model choice, together with an evidence-based understanding of how the engineered features perform on unseen runs.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.3 Dataset and Split Strategy
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The input to this notebook is a feature table in which each row corresponds to a fixed-length window extracted from a sensor run. 
Although modeling is performed on window-level samples, the evaluation design remains <b>run-aware</b>.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Windows originating from the same run are not independent and must not be split across training and validation sets. 
Therefore, group information derived from <code>run_id</code> is treated as a core component of the evaluation strategy rather than a secondary detail.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.4 Evaluation Criteria
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Model comparison in this notebook does not rely on accuracy alone. In addition to overall accuracy, the evaluation emphasizes 
<b>precision</b>, <b>recall</b>, and <b>F1-score</b>, providing a more balanced view of model performance. <i>(Literature Review, Section 5, Evaluation Strategy for Gas Sensor Classification)</i>
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Particular attention is given to <b>recall for the mixture class</b>, as misclassifying mixtures as single-gas samples may negatively impact the downstream Stage 2 classification process.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Results are interpreted through both their average performance and variability across folds, ensuring that the selected model is not only accurate but also stable under run-aware evaluation.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.5 Baseline and Candidate Models
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The modeling workflow begins with a simple and interpretable baseline model, followed by more flexible candidate models. 
This allows us to assess whether the engineered feature space is already linearly separable or whether more complex models are required. <i>(Literature Review, Section 4, Machine Learning Models for Gas Sensor Classification)</i>
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Using a baseline-first strategy also provides a clear reference point, making it easier to interpret performance improvements across models.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
1.6 Model Selection Principle
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The final Stage 1 model is selected based on a combination of predictive performance, consistency across validation folds, and practical suitability for the hierarchical pipeline.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Since Stage 1 acts as a gatekeeper for Stage 2, model selection prioritizes robustness and reliability. 
Errors at this stage may propagate downstream, so decisions are based on stable and leakage-safe evidence rather than isolated performance scores.
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
2. Data Loading
</h2>

<h3 style="color:#2E86C1; font-weight:bold;">
2.1 Load Engineered Feature Dataset
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The first step is to load the final engineered feature dataset produced in the previous notebook. 
This dataset contains metadata and numerical features extracted from fixed-length time-series windows, 
which will be used as input for downstream modeling.
</p>

In [6]:
import pandas as pd

features_df = pd.read_parquet("../data/processed/features_timeseries.parquet")

print("Dataset shape:", features_df.shape)
display(features_df.head())

Dataset shape: (2353, 43)


,run_id,experiment,experiment_folder,run_folder,repeat_index,window_id,start_idx,end_idx,time_start,time_end,...,energy,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position
0,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,0,99,0.000,115.869,...,0.008947,0.415783,2.435652,0.151515,0.193878,0.221913,0.028688,0.000000,1.000000,0.000000
1,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,50,149,58.252,175.526,...,0.007669,-0.536299,-0.479818,0.171717,0.224490,0.108822,0.083578,0.121212,0.878788,0.121212
2,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,100,199,117.016,234.546,...,0.006818,-0.703570,0.285166,0.171717,0.224490,0.032487,0.092399,0.666667,0.333333,0.666667
3,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,150,249,176.640,293.550,...,0.009255,0.342366,-0.604739,0.131313,0.163265,0.082112,0.131995,0.979798,0.020202,0.979798
4,Mixing_LowHigh_concentrations__UV ink-exposed ...,Mixing_LowHigh_concentrations,Mixing_LowHigh concentrastions,..\data\raw\CMOS_addtional_datasets\Mixing_Low...,-1.0,Mixing_LowHigh_concentrations__UV ink-exposed ...,200,299,235.821,352.716,...,0.009037,-0.622299,-0.337232,0.090909,0.122449,0.099987,0.035098,0.474747,0.525253,0.474747


<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The feature-engineered dataset was successfully loaded from the processed data directory. It contains <b>2353 samples</b> and <b>43 columns</b>, 
where each row corresponds to a fixed-length window extracted from the original gas sensor time-series data.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
These windows were generated during preprocessing to transform variable-length raw signals into consistent segments suitable for feature-based machine learning models. 
Each window is represented by a set of engineered features capturing statistical properties, temporal dynamics, shape characteristics, and signal variability.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The dataset includes both metadata and numerical features. Metadata columns such as <b>run_id</b>, <b>experiment</b>, 
<b>window_id</b>, <b>start_idx</b>, <b>end_idx</b>, <b>time_start</b>, and <b>time_end</b> describe the origin and temporal 
position of each window within its corresponding experimental run.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
At this stage, the <b>experiment</b> column serves as the source for defining the target variable (single-gas vs mixture), 
while <b>run_id</b> plays a critical role in maintaining a leakage-safe evaluation strategy. Since multiple windows originate 
from the same run, they are not independent and must be kept within the same subset during training and testing.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, this dataset represents the structured input for Stage 1 modeling. Before training models, an explicit binary target 
variable will be constructed from the experiment labels, and a run-aware splitting strategy will be applied to ensure valid evaluation.
</p>

<h2 style="color:#1B4F72; font-weight:bold;">
3. Dataset Preparation
</h2>

<h3 style="color:#2E86C1; font-weight:bold;">
3.1 Defining Features, Target, and Group Information
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Before training the machine learning models, the dataset is separated into the input feature matrix, the target variable, 
and the grouping information required for leakage-aware evaluation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The target variable is defined as a binary label indicating whether a window corresponds to a <b>mixture-gas</b> or a 
<b>single-gas</b> exposure. This label is derived from the <b>experiment</b> column, where experiments containing the term 
<b>"mixing"</b> are classified as mixture samples, while all other experiments are treated as single-gas samples.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This transformation aligns the dataset with the Stage 1 objective and simplifies the original multi-class problem into a 
binary classification task focused on mixture detection.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Metadata columns such as identifiers, folder paths, and temporal indices are excluded from the feature matrix, as they do 
not represent intrinsic signal characteristics and may introduce unintended bias or leakage. Only numerical features 
derived from the signal are retained for modeling.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The <b>run_id</b> column is preserved separately as grouping information. Since multiple windows originate from the same run, 
they are not independent, and therefore must remain together during training and testing. This grouping strategy is essential 
for ensuring a valid and leakage-safe evaluation.
</p>

In [3]:
# Create binary target: mixture vs single
features_df["target_mixture"] = (
    features_df["experiment"]
    .str.lower()
    .str.contains("mixing")
    .astype(int)
)

# Define target
y = features_df["target_mixture"]

# Preserve group information
groups = features_df["run_id"]

# Remove non-feature and leakage-prone columns
X = features_df.drop(columns=[
    "experiment",
    "target_mixture",
    "run_id",
    "experiment_folder",
    "run_folder",
    "repeat_index",
    "window_id",
    "start_idx",
    "end_idx",
    "time_start",
    "time_end"
])

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of groups:", groups.nunique())

print("\nClass distribution:")
print(y.value_counts())

display(X.head())

Feature matrix shape: (2353, 33)
Target shape: (2353,)
Number of groups: 13

Class distribution:
target_mixture
0    1194
1    1159
Name: count, dtype: int64


,mean,std,min,max,median,range,iqr,cv,first_diff_mean,first_diff_std,...,energy,skewness,kurtosis,flatness,zero_crossing_rate_diff,signal_start,signal_end,peak_to_start_distance,peak_to_end_distance,relative_peak_position
0,0.089142,0.031639,0.020605,0.221913,0.094835,0.201308,0.033844,0.354935,-0.001952,0.008036,...,0.008947,0.415783,2.435652,0.151515,0.193878,0.221913,0.028688,0.000000,1.000000,0.000000
1,0.082972,0.028013,0.020605,0.128674,0.089349,0.108069,0.035378,0.337617,-0.000255,0.004661,...,0.007669,-0.536299,-0.479818,0.171717,0.224490,0.108822,0.083578,0.121212,0.878788,0.121212
2,0.080224,0.019554,0.030791,0.113978,0.082813,0.083187,0.023622,0.243743,0.000605,0.004608,...,0.006818,-0.703570,0.285166,0.171717,0.224490,0.032487,0.092399,0.666667,0.333333,0.666667
3,0.093657,0.021975,0.057616,0.147132,0.090766,0.089516,0.033182,0.234636,0.000504,0.004457,...,0.009255,0.342366,-0.604739,0.131313,0.163265,0.082112,0.131995,0.979798,0.020202,0.979798
4,0.088229,0.035386,0.008150,0.147132,0.091891,0.138982,0.048667,0.401074,-0.000655,0.005779,...,0.009037,-0.622299,-0.337232,0.090909,0.122449,0.099987,0.035098,0.474747,0.525253,0.474747


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The prepared dataset consists of <b>2353 window-level samples</b> described by <b>33 numerical features</b>. These features 
capture statistical, temporal, shape-based, and variability characteristics of the original sensor signals, while all 
non-informative metadata and potential leakage sources have been removed from the feature matrix.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The binary target variable is well balanced across classes, with a nearly equal number of mixture and single-gas samples. 
This balanced distribution supports the use of standard classification metrics without requiring additional resampling strategies.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Although the dataset contains a large number of window-level samples, these originate from a relatively small number of 
experimental runs (<b>13 runs</b>). As a result, windows within the same run are not independent. This makes it essential 
to use <b>group-aware splitting</b> during model evaluation to prevent leakage and ensure realistic performance estimation.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
3.2 Group-Aware Cross-Validation Strategy
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A reliable evaluation strategy is essential for this dataset because multiple window-level samples are derived from the same sensor run. 
Since these samples are not independent, a conventional random split would introduce leakage and produce overly optimistic performance estimates.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The dataset contains only <b>13 unique runs</b>, making a single train-test split potentially unstable and sensitive to the specific runs selected. 
To obtain a more robust estimate of generalization while preserving class balance, this notebook uses <b>StratifiedGroupKFold</b> cross-validation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This strategy keeps all windows from the same <b>run_id</b> within a single fold while approximately preserving the binary class distribution across folds. 
As a result, evaluation remains leakage-aware and more representative of performance on unseen runs.
</p>

In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# Group-aware and class-balanced cross-validation
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("Number of folds:", cv.get_n_splits(X, y, groups))
print("Number of unique runs:", groups.nunique())

Number of folds: 3
Number of unique runs: 13


In [5]:
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y, groups), start=1):
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]
    train_groups = groups.iloc[train_idx].nunique()
    val_groups = groups.iloc[val_idx].nunique()

    print(f"Fold {fold_idx}")
    print(f"  Train groups: {train_groups}, Validation groups: {val_groups}")
    print(f"  Train class distribution:\n{y_train_fold.value_counts(normalize=True).sort_index()}")
    print(f"  Validation class distribution:\n{y_val_fold.value_counts(normalize=True).sort_index()}")
    print("-" * 60)

Fold 1
  Train groups: 9, Validation groups: 4
  Train class distribution:
target_mixture
0    0.474343
1    0.525657
Name: proportion, dtype: float64
  Validation class distribution:
target_mixture
0    0.577483
1    0.422517
Name: proportion, dtype: float64
------------------------------------------------------------
Fold 2
  Train groups: 8, Validation groups: 5
  Train class distribution:
target_mixture
0    0.504698
1    0.495302
Name: proportion, dtype: float64
  Validation class distribution:
target_mixture
0    0.512167
1    0.487833
Name: proportion, dtype: float64
------------------------------------------------------------
Fold 3
  Train groups: 9, Validation groups: 4
  Train class distribution:
target_mixture
0    0.542645
1    0.457355
Name: proportion, dtype: float64
  Validation class distribution:
target_mixture
0    0.429932
1    0.570068
Name: proportion, dtype: float64
------------------------------------------------------------


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The fold inspection confirms that the cross-validation strategy correctly enforces group separation. In each fold, the training and validation subsets contain disjoint sets of <b>run_id</b> values, ensuring that no data leakage occurs between them. This guarantees that the model is always evaluated on entirely unseen runs rather than on related windows from runs already observed during training.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Class distributions across folds remain reasonably balanced, although minor variations are observed between training and validation subsets. This behavior is expected because stratification is applied at the <b>run level</b> rather than the individual window level, and different runs may contribute varying numbers of window samples. As a result, perfect balance cannot always be achieved.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results indicate that the chosen <b>StratifiedGroupKFold</b> configuration provides a well-balanced trade-off between preserving class distribution and strictly enforcing group isolation. This makes the evaluation both leakage-safe and representative of real-world performance on unseen experimental runs.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
3.3 Evaluation Metrics
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To evaluate model performance, multiple classification metrics are used to capture both overall predictive quality and class-specific behavior. 
Since the task is a <b>binary classification problem</b> (single gas vs mixture gas), <b>accuracy</b> is reported as a general performance indicator.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
However, accuracy alone is not sufficient to fully assess model performance. Therefore, additional metrics including <b>precision</b>, 
<b>recall</b>, and <b>F1-score</b> are also used. These metrics provide a more detailed view of classification behavior, particularly 
for the <b>mixture class</b>, which plays a critical role in the hierarchical pipeline.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In this context, <b>recall for the mixture class</b> is especially important, as misclassifying mixture samples as single-gas may 
lead to incorrect routing in Stage 2 and propagate errors through the pipeline.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Under the group-aware cross-validation framework, model performance is evaluated across multiple folds. Final results are reported 
as the <b>mean</b> and <b>standard deviation</b> of each metric across folds, providing a more robust estimate of model generalization 
on unseen runs.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
4.1 Logistic Regression (Baseline Model)
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Logistic Regression is selected as the baseline model to assess whether the engineered features provide sufficient 
discriminative power for distinguishing <b>single-gas</b> and <b>mixture-gas</b> windows. As a linear classifier, 
it offers a simple yet interpretable benchmark for evaluating class separability in the current feature space.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Because Logistic Regression is sensitive to feature scaling, it is implemented within a pipeline that includes 
<b>StandardScaler</b>. This ensures that scaling is performed independently within each cross-validation fold, 
preventing data leakage and maintaining a valid evaluation setup.
</p>

In [7]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Cross-validation with multiple metrics
lr_results = cross_validate(
    lr_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False
)

# Summary
lr_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        lr_results["test_accuracy"].mean(),
        lr_results["test_precision"].mean(),
        lr_results["test_recall"].mean(),
        lr_results["test_f1"].mean()
    ],
    "Std": [
        lr_results["test_accuracy"].std(),
        lr_results["test_precision"].std(),
        lr_results["test_recall"].std(),
        lr_results["test_f1"].std()
    ]
})

display(lr_summary)

,Metric,Mean,Std
0,Accuracy,0.915470,0.035924
1,Precision,0.926719,0.051060
2,Recall,0.907656,0.064804
3,F1-score,0.914724,0.034277


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The baseline Logistic Regression model achieves strong performance across all evaluation metrics, with an average 
accuracy of approximately <b>91.5%</b> and a balanced F1-score close to <b>0.91</b>. This indicates that the engineered 
feature space already provides a high level of linear separability between single-gas and mixture-gas samples.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The <b>precision</b> score suggests that when the model predicts mixture samples, it is correct in most cases. At the 
same time, the <b>recall</b> value (approximately 0.91) indicates that the majority of mixture windows are successfully 
identified, which is particularly important for the reliability of the hierarchical pipeline.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The relatively low standard deviation across folds confirms that the model performance is stable under the 
group-aware cross-validation setting, suggesting consistent generalization across different runs.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, these results indicate that even a simple linear model can effectively separate the two classes, providing 
a strong baseline for comparison with more complex non-linear models in subsequent steps.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
4.2 Support Vector Machine (SVM)
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Support Vector Machines (SVM) are evaluated as a more flexible alternative to the linear baseline model. Using the 
<b>Radial Basis Function (RBF)</b> kernel, SVM can capture non-linear relationships and model more complex decision boundaries 
within the engineered feature space.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The purpose of this experiment is to assess whether introducing a non-linear classifier improves the separation between 
<b>single-gas</b> and <b>mixture-gas</b> samples beyond the baseline Logistic Regression model.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
As with Logistic Regression, SVM is sensitive to feature scaling. Therefore, it is implemented within a pipeline that includes 
<b>StandardScaler</b>, ensuring that scaling is applied independently within each cross-validation fold and preventing leakage.
</p>

In [8]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
import pandas as pd

# SVM pipeline
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", random_state=42))
])

# Cross-validation with multiple metrics
svm_results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False
)

# Summary table
svm_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        svm_results["test_accuracy"].mean(),
        svm_results["test_precision"].mean(),
        svm_results["test_recall"].mean(),
        svm_results["test_f1"].mean()
    ],
    "Std": [
        svm_results["test_accuracy"].std(),
        svm_results["test_precision"].std(),
        svm_results["test_recall"].std(),
        svm_results["test_f1"].std()
    ]
})

display(svm_summary)

,Metric,Mean,Std
0,Accuracy,0.874368,0.066295
1,Precision,0.959706,0.020010
2,Recall,0.793935,0.141320
3,F1-score,0.860874,0.071555


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The SVM model achieves moderate overall performance, with an average accuracy of approximately <b>87.4%</b> and an F1-score of about <b>0.86</b>. 
Compared with the Logistic Regression baseline, this represents a clear reduction in overall classification quality.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Although the model shows very high <b>precision</b> (approximately <b>0.96</b>), its <b>recall</b> is substantially lower (approximately <b>0.79</b>). 
This indicates that when SVM predicts the mixture class, it is usually correct, but it fails to identify a considerable portion of true mixture samples.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
From the perspective of the hierarchical pipeline, this behavior is less desirable than the Logistic Regression baseline. 
In Stage 1, missing mixture samples is costly because they may be incorrectly routed into the single-gas branch, which can negatively affect downstream classification.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The relatively larger standard deviation, especially for <b>recall</b>, also suggests that SVM is less stable across folds under the current run-aware evaluation setup. 
Overall, the results indicate that the additional non-linearity of the RBF kernel does not provide an advantage here and may reduce robustness compared with the simpler linear baseline.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
4.3 Random Forest
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Random Forest is evaluated as an ensemble-based model that combines multiple decision trees to improve predictive 
performance and robustness. Unlike linear models, it can naturally capture non-linear relationships and complex 
feature interactions without requiring explicit feature transformations.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This model is particularly suitable for the current dataset, where the engineered features capture diverse aspects 
of the signal, including temporal dynamics, statistical behavior, and structural patterns. Random Forest can effectively 
leverage these heterogeneous features to learn flexible and high-quality decision boundaries for separating 
<b>single-gas</b> and <b>mixture-gas</b> samples.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition, Random Forest is less sensitive to feature scaling and tends to be robust to noise and variability across runs, 
making it a strong candidate for comparison with both linear and kernel-based models.
</p>

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
import pandas as pd

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Cross-validation with multiple metrics
rf_results = cross_validate(
    rf_model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False,
    n_jobs=-1
)

# Summary table
rf_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        rf_results["test_accuracy"].mean(),
        rf_results["test_precision"].mean(),
        rf_results["test_recall"].mean(),
        rf_results["test_f1"].mean()
    ],
    "Std": [
        rf_results["test_accuracy"].std(),
        rf_results["test_precision"].std(),
        rf_results["test_recall"].std(),
        rf_results["test_f1"].std()
    ]
})

display(rf_summary)

,Metric,Mean,Std
0,Accuracy,0.961046,0.023452
1,Precision,0.974578,0.016048
2,Recall,0.951272,0.046269
3,F1-score,0.961912,0.020644


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The Random Forest model achieves the strongest performance among the evaluated models, with an average accuracy of approximately 
<b>96.1%</b> and an F1-score of about <b>0.96</b>. This represents a clear improvement over both the Logistic Regression and SVM models.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Both <b>precision</b> and <b>recall</b> are consistently high, indicating that the model is effective at correctly identifying 
mixture samples while also minimizing false positives. In particular, the recall value (approximately <b>0.95</b>) suggests that 
the majority of mixture windows are successfully detected, which is critical for maintaining reliability in the hierarchical pipeline.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The relatively low standard deviation across all metrics indicates stable performance across folds, suggesting that the model 
generalizes well to unseen runs under the group-aware evaluation setting.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, these results demonstrate that modeling non-linear feature interactions provides a clear advantage for this task. 
Random Forest effectively captures the complexity of the engineered feature space and emerges as a strong candidate for 
Stage 1 model selection.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
4.4 Gradient Boosting
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Gradient Boosting is evaluated as a sequential ensemble method in which each new tree attempts to correct the errors 
made by the previous ones. Unlike Random Forest, which builds trees independently, Gradient Boosting refines the model 
iteratively and can capture complex decision boundaries through this process.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This approach is relevant for the current task because the engineered features may contain subtle non-linear interactions 
related to temporal dynamics, signal shape, and statistical properties. By focusing on difficult samples during training, 
Gradient Boosting aims to improve classification performance beyond simpler models.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
As an ensemble model, it is also capable of handling complex feature relationships and variability across runs, making it 
a suitable candidate for comparison with Random Forest and other previously evaluated models.
</p>

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
import pandas as pd

# Gradient Boosting model
gb_model = GradientBoostingClassifier(random_state=42)

# Cross-validation with multiple metrics
gb_results = cross_validate(
    gb_model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False,
    n_jobs=-1
)

# Summary table
gb_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        gb_results["test_accuracy"].mean(),
        gb_results["test_precision"].mean(),
        gb_results["test_recall"].mean(),
        gb_results["test_f1"].mean()
    ],
    "Std": [
        gb_results["test_accuracy"].std(),
        gb_results["test_precision"].std(),
        gb_results["test_recall"].std(),
        gb_results["test_f1"].std()
    ]
})

display(gb_summary)

,Metric,Mean,Std
0,Accuracy,0.950089,0.023130
1,Precision,0.950734,0.033951
2,Recall,0.953666,0.040359
3,F1-score,0.951151,0.020567


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The Gradient Boosting model achieves strong overall performance, with an average accuracy of approximately <b>95.0%</b> 
and an F1-score close to <b>0.95</b>. This confirms that ensemble-based approaches are well suited for modeling the 
engineered feature space.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Both <b>precision</b> and <b>recall</b> remain high and well balanced, indicating that the model is effective at identifying 
mixture samples while maintaining a low rate of false positives. This balance is particularly important for ensuring 
reliable decision-making in the hierarchical classification pipeline.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
However, when compared with the Random Forest model, Gradient Boosting does not provide a clear improvement in performance. 
While results are competitive, the slight decrease in accuracy and F1-score suggests that the additional sequential 
complexity does not translate into a meaningful gain for this dataset.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, these results indicate that although Gradient Boosting is a strong and reliable model, Random Forest remains the 
best-performing approach among the evaluated candidates for Stage 1 classification.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
Note on Additional Ensemble Models
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
More advanced boosting-based models such as <b>XGBoost</b> and <b>LightGBM</b> are well known for their strong performance 
on structured tabular data and could potentially further improve classification results by modeling complex non-linear 
interactions more effectively.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
However, since the current experiments already demonstrate strong and stable performance using classical models, the focus 
of this stage remains on establishing a reliable and interpretable baseline under a leakage-aware evaluation framework. 
Exploration of more advanced ensemble methods is therefore deferred to a later optimization phase if further improvements 
are required.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
4.5 Model Comparison
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To systematically compare the evaluated models, cross-validation results are summarized using <b>accuracy</b> and 
<b>F1-score</b>, along with their corresponding standard deviations. This comparison captures both predictive performance 
and stability across folds, which is particularly important given the variability between sensor runs.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
All models are evaluated under the same group-aware cross-validation framework, ensuring a fair comparison and allowing 
reliable conclusions about generalization performance on unseen runs.
</p>

In [13]:
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Mean Accuracy": [
        lr_results["test_accuracy"].mean(),
        svm_results["test_accuracy"].mean(),
        rf_results["test_accuracy"].mean(),
        gb_results["test_accuracy"].mean()
    ],
    "Mean F1-score": [
        lr_results["test_f1"].mean(),
        svm_results["test_f1"].mean(),
        rf_results["test_f1"].mean(),
        gb_results["test_f1"].mean()
    ],
    "Std Accuracy": [
        lr_results["test_accuracy"].std(),
        svm_results["test_accuracy"].std(),
        rf_results["test_accuracy"].std(),
        gb_results["test_accuracy"].std()
    ]
})

comparison_df = comparison_df.sort_values(
    by="Mean F1-score",
    ascending=False
)

display(comparison_df)

,Model,Mean Accuracy,Mean F1-score,Std Accuracy
2,Random Forest,0.961046,0.961912,0.023452
3,Gradient Boosting,0.950089,0.951151,0.023130
0,Logistic Regression,0.915470,0.914724,0.035924
1,SVM,0.874368,0.860874,0.066295


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The comparison results show a clear performance ranking among the evaluated models. <b>Random Forest</b> achieves the highest 
accuracy and F1-score, indicating the best overall classification performance. It also maintains low standard deviation, 
suggesting stable behavior across different folds.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
<b>Gradient Boosting</b> performs similarly but does not surpass Random Forest, indicating that additional model complexity 
does not lead to a meaningful improvement for this dataset. Both ensemble methods significantly outperform the linear and 
kernel-based models.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
<b>Logistic Regression</b> provides strong baseline performance, demonstrating that the engineered feature space is already 
largely separable. However, its performance remains below that of ensemble methods, suggesting that non-linear feature 
interactions contribute to improved classification.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
<b>SVM</b> shows the weakest performance among the evaluated models, with lower recall and higher variability across folds. 
This indicates reduced robustness and less reliable generalization under the current run-aware evaluation setup.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, <b>Random Forest</b> is selected as the preferred model for Stage 1, as it provides the best balance between accuracy, 
recall, and stability. Its ability to consistently identify mixture samples while maintaining low variance makes it particularly 
well suited for the hierarchical classification pipeline.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
4.6 Neural Network (MLP)
</h3>

<h3 style="color:#2E86C1; font-weight:bold;">
Note on Neural Network Convergence and Model Suitability
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A Multi-Layer Perceptron (MLP) is evaluated as a neural network-based approach to assess whether a more flexible model 
can further improve classification performance. MLPs are capable of learning complex non-linear relationships, but their 
effectiveness strongly depends on the amount of available training data and the stability of patterns across groups.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Given the relatively small number of unique runs in the current dataset, the MLP model is included primarily as a comparative 
experiment rather than a core candidate for final model selection. Its performance helps evaluate whether neural-network-based 
approaches offer a meaningful advantage over classical machine learning models in this setting.
</p>

In [14]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# MLP pipeline
mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=500,
        random_state=42
    ))
])

# Cross-validation with multiple metrics
mlp_results = cross_validate(
    mlp_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False,
    n_jobs=-1
)

# Summary table
mlp_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        mlp_results["test_accuracy"].mean(),
        mlp_results["test_precision"].mean(),
        mlp_results["test_recall"].mean(),
        mlp_results["test_f1"].mean()
    ],
    "Std": [
        mlp_results["test_accuracy"].std(),
        mlp_results["test_precision"].std(),
        mlp_results["test_recall"].std(),
        mlp_results["test_f1"].std()
    ]
})

display(mlp_summary)

,Metric,Mean,Std
0,Accuracy,0.928444,0.044065
1,Precision,0.913852,0.060319
2,Recall,0.954480,0.033826
3,F1-score,0.932734,0.040440


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The MLP model achieves solid performance, with an average accuracy of approximately <b>92.8%</b> and an F1-score of about 
<b>0.93</b>. These results indicate that the neural network is capable of capturing relevant patterns within the engineered 
feature space.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The model demonstrates relatively high <b>recall</b> (approximately <b>0.95</b>), suggesting that it is effective at identifying 
mixture samples. However, its <b>precision</b> is slightly lower compared to ensemble-based models, indicating a higher rate 
of false positives.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Despite its competitive performance, the MLP does not outperform the best-performing ensemble models, particularly Random Forest. 
In addition, neural networks typically require larger datasets to achieve stable and reliable generalization, which is a limitation 
in the current setting with only a small number of unique runs.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Training stability is also a consideration, as neural networks may require careful tuning and sufficient iterations to converge properly. 
Under the current configuration, the model provides useful insights but does not offer a clear advantage over simpler and more robust 
approaches.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, while the MLP confirms that the feature space contains learnable non-linear structure, it is not selected as a primary 
model for Stage 1. Preference is given to ensemble-based methods, which provide stronger performance and more stable behavior 
under the group-aware evaluation framework.
</p>

<h2 style="color:#2E86C1; font-weight:bold;">
5. PCA-Based Experiment
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In addition to model comparison, an auxiliary experiment is conducted to evaluate the impact of <b>dimensionality reduction</b> 
on classification performance. Since the engineered feature set includes statistical, temporal, and shape-based descriptors, 
some degree of feature correlation and redundancy is expected.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To investigate this, <b>Principal Component Analysis (PCA)</b> is applied to transform the original feature space into a 
lower-dimensional representation while preserving most of the variance. This allows assessment of whether reducing feature 
redundancy can improve generalization or model efficiency.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The experiment focuses on models that are sensitive to feature scaling and feature-space geometry, particularly 
<b>Logistic Regression</b> and <b>Support Vector Machine (SVM)</b>. Tree-based models are excluded, as they are generally 
robust to correlated features and do not typically benefit from PCA transformations.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
5.1 Logistic Regression with PCA
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To examine the effect of dimensionality reduction on a linear classifier, PCA is applied prior to Logistic Regression. 
The objective of this experiment is to assess whether reducing feature redundancy and transforming the feature space into 
orthogonal components improves classification performance.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
PCA is configured to retain <b>95% of the total variance</b>, ensuring that most of the information from the original 
feature set is preserved while reducing dimensionality. This setup allows evaluation of whether a more compact 
representation benefits the linear baseline model.
</p>

In [15]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Logistic Regression + PCA pipeline
lr_pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Cross-validation with multiple metrics
lr_pca_results = cross_validate(
    lr_pca_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False
)

# Summary table
lr_pca_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        lr_pca_results["test_accuracy"].mean(),
        lr_pca_results["test_precision"].mean(),
        lr_pca_results["test_recall"].mean(),
        lr_pca_results["test_f1"].mean()
    ],
    "Std": [
        lr_pca_results["test_accuracy"].std(),
        lr_pca_results["test_precision"].std(),
        lr_pca_results["test_recall"].std(),
        lr_pca_results["test_f1"].std()
    ]
})

display(lr_pca_summary)

,Metric,Mean,Std
0,Accuracy,0.871617,0.010215
1,Precision,0.881111,0.080098
2,Recall,0.874534,0.095504
3,F1-score,0.869167,0.007776


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The application of PCA leads to a noticeable reduction in performance for Logistic Regression, with accuracy decreasing 
to approximately <b>87.2%</b> and the F1-score dropping to around <b>0.87</b>. Compared to the original feature space, this 
represents a clear degradation in classification quality.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This result suggests that the original engineered features already provide a well-structured and informative representation 
for the classification task. By transforming the feature space into principal components, some discriminative information 
relevant for separating <b>single-gas</b> and <b>mixture-gas</b> samples may be lost.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Although PCA reduces feature redundancy and enforces orthogonality, it does not necessarily preserve class-separating 
structures. In this case, the transformation appears to remove useful signal characteristics that were effectively 
utilized by the original Logistic Regression model.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, these results indicate that dimensionality reduction via PCA is not beneficial for the current feature space. 
The original engineered features already strike a good balance between expressiveness and redundancy, and are better 
suited for linear classification without transformation.
</p>

<h3 style="color:#2E86C1; font-weight:bold;">
5.2 Support Vector Machine with PCA
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To further investigate the effect of dimensionality reduction, PCA is applied prior to the Support Vector Machine (SVM) model. 
Because SVM is sensitive to feature scaling and the geometric structure of the feature space, transforming the original 
features into orthogonal principal components may influence class separation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
By retaining <b>95% of the total variance</b>, PCA preserves most of the information in the data while reducing redundancy 
and potential noise. This experiment evaluates whether a more compact representation can improve the performance of the 
non-linear SVM classifier.
</p>

In [16]:
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import pandas as pd

# SVM + PCA pipeline
svm_pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", SVC(kernel="rbf"))
])

# Cross-validation with multiple metrics
svm_pca_results = cross_validate(
    svm_pca_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    return_train_score=False
)

# Summary table
svm_pca_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean": [
        svm_pca_results["test_accuracy"].mean(),
        svm_pca_results["test_precision"].mean(),
        svm_pca_results["test_recall"].mean(),
        svm_pca_results["test_f1"].mean()
    ],
    "Std": [
        svm_pca_results["test_accuracy"].std(),
        svm_pca_results["test_precision"].std(),
        svm_pca_results["test_recall"].std(),
        svm_pca_results["test_f1"].std()
    ]
})

display(svm_pca_summary)

,Metric,Mean,Std
0,Accuracy,0.862976,0.053983
1,Precision,0.946459,0.051570
2,Recall,0.787047,0.148372
3,F1-score,0.847288,0.059325


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The application of PCA results in a further decrease in performance for the SVM model, with accuracy dropping to approximately 
<b>86.3%</b> and the F1-score declining to around <b>0.85</b>. This represents a clear degradation compared to both the original 
SVM model and the Logistic Regression baseline.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Although the model maintains relatively high <b>precision</b>, the <b>recall</b> is significantly lower (approximately <b>0.78</b>), 
indicating that a substantial number of mixture samples are not correctly identified. This behavior is particularly undesirable 
for Stage 1, where missing mixture cases can negatively affect downstream classification.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
These results suggest that PCA does not improve the geometric structure of the feature space for SVM. Instead, the transformation 
appears to remove important discriminative information that was present in the original engineered features.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the PCA-based representation is not beneficial for SVM in this context. The original feature space provides better 
separability and leads to more reliable performance under the group-aware evaluation framework.
</p>

<h3 style="color:#2874A6; font-weight:bold;">
5.3 Final Comparison: Baseline vs PCA
</h3>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To evaluate the effect of dimensionality reduction, the performance of the PCA-based models is compared directly 
with their corresponding baseline versions. This comparison helps determine whether reducing the dimensionality of 
the feature space improves classification performance or whether the original engineered features already provide a 
sufficiently informative representation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The analysis focuses on <b>Logistic Regression</b> and <b>SVM</b>, as both models are sensitive to feature scaling 
and the geometric structure of the feature space. Comparing baseline and PCA-enhanced versions allows a direct 
assessment of the usefulness of dimensionality reduction for this task.
</p>

In [17]:
pca_comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Logistic Regression + PCA",
        "SVM",
        "SVM + PCA"
    ],
    "Mean Accuracy": [
        lr_results["test_accuracy"].mean(),
        lr_pca_results["test_accuracy"].mean(),
        svm_results["test_accuracy"].mean(),
        svm_pca_results["test_accuracy"].mean()
    ],
    "Mean F1-score": [
        lr_results["test_f1"].mean(),
        lr_pca_results["test_f1"].mean(),
        svm_results["test_f1"].mean(),
        svm_pca_results["test_f1"].mean()
    ],
    "Std Accuracy": [
        lr_results["test_accuracy"].std(),
        lr_pca_results["test_accuracy"].std(),
        svm_results["test_accuracy"].std(),
        svm_pca_results["test_accuracy"].std()
    ]
})

display(pca_comparison_df)

,Model,Mean Accuracy,Mean F1-score,Std Accuracy
0,Logistic Regression,0.915470,0.914724,0.035924
1,Logistic Regression + PCA,0.871617,0.869167,0.010215
2,SVM,0.874368,0.860874,0.066295
3,SVM + PCA,0.862976,0.847288,0.053983


<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The comparison between baseline models and their PCA-enhanced counterparts shows that dimensionality reduction does not 
improve performance for this task. In both cases, applying PCA leads to a noticeable decrease in accuracy and F1-score.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
For <b>Logistic Regression</b>, the accuracy drops from approximately <b>91.5%</b> to <b>87.2%</b>, and the F1-score decreases 
accordingly. A similar trend is observed for <b>SVM</b>, where both accuracy and F1-score decline after applying PCA. 
These results indicate that the dimensionality reduction process removes information that is important for class separation.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This behavior suggests that the engineered features are already well-structured and do not suffer significantly from redundancy 
or multicollinearity. Instead of improving generalization, PCA appears to distort the original feature space and reduce the 
discriminative power of the models.
</p>

<p style="font-size:16px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results demonstrate that <b>dimensionality reduction is not beneficial in this context</b>. The original 
feature space provides a more effective representation for distinguishing between <b>single-gas</b> and <b>mixture-gas</b> samples, 
and therefore PCA is not considered for further modeling stages.
</p>

<h2 style="color:#2E86C1; font-weight:bold;">
6. Key Findings and Next Step
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
This study investigated the classification of time-series gas sensor data into <b>single-gas</b> and 
<b>mixture-gas</b> categories using engineered features and multiple machine learning models under a 
leakage-aware evaluation framework. The results demonstrate that transforming raw sensor signals into structured 
feature representations enables highly effective classification using classical machine learning approaches.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
A key finding of this work is that the engineered feature space is highly informative. Even the linear baseline 
model, <b>Logistic Regression</b>, achieved strong performance, indicating that the feature engineering stage 
successfully captured important temporal, statistical, and shape-based characteristics of the sensor signals.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Among all evaluated models, <b>Random Forest</b> achieved the best overall performance in terms of both accuracy 
and F1-score, while also demonstrating stable results across folds. This suggests that incorporating non-linear 
feature interactions provides additional predictive power beyond the linear baseline.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The PCA-based experiments further confirmed that dimensionality reduction does not provide a meaningful improvement. 
Applying PCA resulted in decreased performance for both Logistic Regression and SVM, indicating that the original 
feature space already provides a well-structured and information-rich representation.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Another important observation is the variability in performance across folds, which reflects differences between 
experimental runs. This highlights the importance of using <b>group-aware cross-validation</b> to obtain realistic 
and reliable performance estimates for unseen runs.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
Overall, the results indicate that successful classification in this study depends on strong feature engineering 
combined with robust leakage-aware validation. As the next step, the <b>Random Forest model</b> is selected as the 
primary Stage 1 classifier, and will be used for further analysis, including confusion matrix evaluation, error 
analysis, and integration into the hierarchical classification pipeline.
</p>

<h2 style="color:#2E86C1; font-weight:bold;">
7. Preparing Single-Gas Subset for Stage-2 Modeling
</h2>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
To support the next stage of the hierarchical modeling framework, a dedicated subset containing only 
<b>true single-gas samples</b> is prepared from the feature-engineered dataset. This step is necessary because the 
Stage-2 modeling task focuses specifically on distinguishing between the 
<b>Toluene</b> and <b>2-butanone</b> classes.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
At this stage, the subset is constructed using the <b>ground-truth labels</b> rather than the predictions of the 
Stage-1 model. This design choice ensures that the Stage-2 classifier is trained on correctly labeled samples and 
is not affected by potential misclassifications introduced in the first-stage model. In other words, this approach 
isolates the performance of Stage-2 and prevents error propagation across the hierarchical pipeline.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
In a production setting, the output of Stage-1 would be used to filter samples before applying the Stage-2 model. 
However, during model development and evaluation, using ground-truth labels provides a cleaner and more reliable 
assessment of Stage-2 performance.
</p>

<p style="font-size:17px; margin-left:25px; line-height:1.7; max-width:900px; text-align: justify;">
The resulting single-gas subset is saved as a separate processed dataset to enable modular reuse in the next notebook. 
This separation supports a cleaner workflow and aligns with a production-oriented pipeline design.
</p>

In [19]:
# Create binary target for single vs mixture
features_df["target_mixture"] = features_df["experiment"].apply(
    lambda x: 1 if "mixing" in str(x).lower() else 0
)

# Keep only true single-gas samples for stage-2 modeling
single_df = features_df[features_df["target_mixture"] == 0].copy()

print("Single-gas subset shape:", single_df.shape)
print("Unique experiments in single subset:")
print(single_df["experiment"].value_counts())

# Save for next notebook
single_df.to_parquet("../data/processed/features_single_gas.parquet", index=False)

print("Saved single-gas feature file to ../data/processed/features_single_gas.parquet")

Single-gas subset shape: (1194, 44)
Unique experiments in single subset:
experiment
UV ink-Toluene           315
ZIF8 Layer - Toluene     306
ZIF8 ink - 2-butanone    290
UV ink - 2-butanone      283
Name: count, dtype: int64
Saved single-gas feature file to ../data/processed/features_single_gas.parquet
